# Multilingual Prompt-Attack Classifier
**Chosen approved model:** `aisuko/distilbert-base-multilingual-cased` (the Kaggle-provided DistilBERT multilingual cased model).

Fine-tunes a three-class sequence classifier, uses inverse-frequency class weights for imbalance, evaluates on a stratified validation split, then predicts every row in `test.csv` and writes `submission.csv` (`id,label`). Run in Kaggle with the approved model dataset attached and the provided CSV files available in the working directory.

In [ ]:
from pathlib import Path
import os, random
import numpy as np
import pandas as pd
import torch
from sklearn.model_selection import train_test_split
from sklearn.metrics import classification_report, f1_score
from transformers import (AutoTokenizer, AutoModelForSequenceClassification,
                          Trainer, TrainingArguments, DataCollatorWithPadding)

SEED = 42
LABELS = ['BENIGN', 'JAILBREAK', 'PROMPT_INJECTION']
LABEL2ID = {name: i for i, name in enumerate(LABELS)}
ID2LABEL = {i: name for name, i in LABEL2ID.items()}
random.seed(SEED); np.random.seed(SEED); torch.manual_seed(SEED)
if torch.cuda.is_available(): torch.cuda.manual_seed_all(SEED)


In [ ]:
def find_csv(name):
    candidates = [Path.cwd() / name, Path('/kaggle/working') / name]
    for path in candidates:
        if path.is_file(): return path
    raise FileNotFoundError(f'{name} not found; place it in the notebook working directory.')

train_df = pd.read_csv(find_csv('train.csv'))
test_df = pd.read_csv(find_csv('test.csv'))
required_train = {'id', 'text', 'label'}
required_test = {'id', 'text'}
if not required_train.issubset(train_df.columns): raise ValueError(f'train.csv must contain {required_train}')
if not required_test.issubset(test_df.columns): raise ValueError(f'test.csv must contain {required_test}')
if train_df[list(required_train)].isnull().any().any(): raise ValueError('Training id, text, and label cannot be missing.')
if test_df[['id', 'text']].isnull().any().any(): raise ValueError('Test id and text cannot be missing.')
train_df['label'] = train_df['label'].astype(str).str.strip().str.upper()
unknown = sorted(set(train_df.label) - set(LABELS))
if unknown: raise ValueError(f'Unexpected training labels: {unknown}')
if set(train_df.label) != set(LABELS): raise ValueError('Training data must contain all three classes.')
train_df['text'] = train_df['text'].astype(str)
test_df['text'] = test_df['text'].astype(str)
# Explicitly expose the provided language metadata to the classifier.
if 'language' in train_df.columns and 'language' in test_df.columns:
    train_df['text'] = '[LANG=' + train_df['language'].fillna('unknown').astype(str) + '] ' + train_df['text']
    test_df['text'] = '[LANG=' + test_df['language'].fillna('unknown').astype(str) + '] ' + test_df['text']
print(f'Train: {len(train_df):,} rows; test: {len(test_df):,} rows')
print(train_df['label'].value_counts(normalize=True).reindex(LABELS))

In [ ]:
# Kaggle mounts attached datasets below /kaggle/input. Locate model files without
# relying on a potentially unavailable network download.
MODEL_NAME = 'aisuko/distilbert-base-multilingual-cased'
model_dirs = []
kaggle_input = Path('/kaggle/input')
if kaggle_input.exists():
    model_dirs = [p.parent for p in kaggle_input.rglob('config.json')
                  if p.parent.is_dir() and ('distilbert' in str(p.parent).lower() or
                     'multilingual' in str(p.parent).lower())]
MODEL_PATH = str(model_dirs[0]) if model_dirs else MODEL_NAME
print('Loading approved backbone from:', MODEL_PATH)
tokenizer = AutoTokenizer.from_pretrained(MODEL_PATH)
model = AutoModelForSequenceClassification.from_pretrained(
    MODEL_PATH, num_labels=len(LABELS), label2id=LABEL2ID, id2label=ID2LABEL,
    ignore_mismatched_sizes=True)


In [ ]:
train_part, valid_part = train_test_split(
    train_df, test_size=0.12, random_state=SEED, stratify=train_df['label'])
class_counts = train_part['label'].value_counts().reindex(LABELS).to_numpy()
class_weights = len(train_part) / (len(LABELS) * class_counts)
class_weights = torch.tensor(class_weights, dtype=torch.float32)

class TextDataset(torch.utils.data.Dataset):
    def __init__(self, frame, with_labels):
        self.encodings = tokenizer(frame['text'].tolist(), truncation=True, max_length=256)
        self.labels = [LABEL2ID[x] for x in frame['label']] if with_labels else None
    def __len__(self): return len(self.encodings['input_ids'])
    def __getitem__(self, idx):
        item = {key: torch.tensor(values[idx]) for key, values in self.encodings.items()}
        if self.labels is not None: item['labels'] = torch.tensor(self.labels[idx], dtype=torch.long)
        return item

train_ds = TextDataset(train_part, True)
valid_ds = TextDataset(valid_part, True)
test_ds = TextDataset(test_df, False)
def compute_metrics(eval_pred):
    logits, labels = eval_pred
    preds = np.argmax(logits, axis=-1)
    return {'macro_f1': f1_score(labels, preds, average='macro')}

class WeightedTrainer(Trainer):
    def compute_loss(self, model, inputs, return_outputs=False, num_items_in_batch=None):
        labels = inputs.pop('labels')
        outputs = model(**inputs)
        loss = torch.nn.functional.cross_entropy(
            outputs.logits, labels, weight=class_weights.to(outputs.logits.device))
        return (loss, outputs) if return_outputs else loss


In [ ]:
args = TrainingArguments(
    output_dir='./attack_classifier_checkpoints',
    learning_rate=2e-5, num_train_epochs=3, weight_decay=0.01,
    per_device_train_batch_size=16, per_device_eval_batch_size=32,
    gradient_accumulation_steps=1, warmup_ratio=0.06,
    eval_strategy='epoch', save_strategy='epoch',
    load_best_model_at_end=True, metric_for_best_model='macro_f1',
    greater_is_better=True, save_total_limit=2,
    logging_steps=100, report_to='none', seed=SEED,
    fp16=torch.cuda.is_available(),
   )
trainer = WeightedTrainer(
    model=model, args=args, train_dataset=train_ds, eval_dataset=valid_ds,
    data_collator=DataCollatorWithPadding(tokenizer=tokenizer),
    compute_metrics=compute_metrics)
trainer.train()
metrics = trainer.evaluate()
print('Validation metrics:', metrics)
valid_logits = trainer.predict(valid_ds).predictions
valid_preds = np.argmax(valid_logits, axis=-1)
print(classification_report([LABEL2ID[x] for x in valid_part.label], valid_preds,
      labels=list(range(len(LABELS))), target_names=LABELS, zero_division=0))

In [ ]:
# Produce exactly one prediction per test id and preserve test-row ordering.
test_logits = trainer.predict(test_ds).predictions
test_labels = [ID2LABEL[int(i)] for i in np.argmax(test_logits, axis=-1)]
submission = pd.DataFrame({'id': test_df['id'], 'label': test_labels})
if len(submission) != len(test_df) or submission['id'].isna().any():
    raise RuntimeError('Submission row/id validation failed.')
if not set(submission['label']).issubset(set(LABELS)):
    raise RuntimeError('Submission contains an invalid label.')
output_path = Path.cwd() / 'submission.csv'
submission.to_csv(output_path, index=False)
# Verify the written artifact and required schema.
check = pd.read_csv(output_path)
assert check.columns.tolist() == ['id', 'label']
assert len(check) == len(test_df) and check['id'].tolist() == test_df['id'].tolist()
print(f'Wrote {output_path} with {len(check):,} predictions.')
display(check.head())